## Creating Tools and Building Agents

Design a simple application that can interpret natural language math queries and solve them by using modular tools. Specifically, the application should support operations such as adding two numbers and computing the square of a number. The system should be implemented using both static Chains for predefined logic and dynamic Agents for flexible, query-driven tool selection.

In [12]:
import os
import getpass
#os.environ["LANGSMITH_API_KEY"] = getpass.getpass("Enter your LangSmith API key: ")


Enter your LangSmith API key:  ········


In [3]:
from langchain_core.tools import tool
from langchain import hub
from langchain.agents import AgentExecutor, create_tool_calling_agent

@tool
def add_numbers(a: int, b: int) -> int:
    """Add two numbers and return the result."""
    return a + b

@tool
def square_number(x: int) -> int:
    """Return the square of a number."""
    return x*x


In [4]:
print(add_numbers.name)
print(add_numbers.description)
print(add_numbers.args)

add_numbers
Add two numbers and return the result.
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [5]:
add_numbers.invoke({"a":111117, "b": 5})

111122

Tool Usage with Chains:

In LangChain, a chain is used when the number of tool invocations is predetermined. For example, a simple chain can be created to multiply two numbers entered by the user. This involves using a tool for multiplication in a fixed sequence of steps. In this case, the chain would take the user’s inputs, multiply them using the specified tool, and return the result.

#### Tool or function calling allows language models to use external tools by invoking APIs, often called function calls.

In [6]:
# AWS Bedrock LLM Setup
# ---------------------------
from langchain_aws import ChatBedrockConverse, BedrockEmbeddings
#llm = ChatBedrockConverse(model_id="amazon.nova-lite-v1:0", region_name="us-east-1", temperature=0.5, max_tokens=50)
llm = ChatBedrockConverse(model_id="cohere.command-r-plus-v1:0", region_name="us-east-1", temperature=0.5, max_tokens=500)
llm.invoke("Hi")

AIMessage(content='Hello! How can I help you today?', additional_kwargs={}, response_metadata={'ResponseMetadata': {'RequestId': 'b70cfc4a-6cea-4966-83af-ecf724ccf0a8', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Mon, 09 Mar 2026 06:21:52 GMT', 'content-type': 'application/json', 'content-length': '232', 'connection': 'keep-alive', 'x-amzn-requestid': 'b70cfc4a-6cea-4966-83af-ecf724ccf0a8'}, 'RetryAttempts': 0}, 'stopReason': 'end_turn', 'metrics': {'latencyMs': [394]}, 'model_name': 'cohere.command-r-plus-v1:0'}, id='run--8f9c23ae-e10c-4ff2-994e-6a5a9944255b-0', usage_metadata={'input_tokens': 1, 'output_tokens': 9, 'total_tokens': 10, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}})

In [9]:
# Bind both tools
llm_with_tools = llm.bind_tools([add_numbers,square_number])

In [10]:
llm_with_tools

RunnableBinding(bound=ChatBedrockConverse(client=<botocore.client.BedrockRuntime object at 0x73136e70feb0>, bedrock_client=<botocore.client.Bedrock object at 0x73136ddef010>, model_id='cohere.command-r-plus-v1:0', max_tokens=500, temperature=0.5, region_name='us-east-1', provider='cohere', supports_tool_choice_values=()), kwargs={'tools': [{'type': 'function', 'function': {'name': 'add_numbers', 'description': 'Add two numbers and return the result.', 'parameters': {'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'}}}, {'type': 'function', 'function': {'name': 'square_number', 'description': 'Return the square of a number.', 'parameters': {'properties': {'x': {'type': 'integer'}}, 'required': ['x'], 'type': 'object'}}}]}, config={}, config_factories=[])

In [8]:
from operator import itemgetter

# Chain: Ask to add -> extract args -> call add_numbers -> wrap result -> ask to square -> extract args -> call square_number

chain = (
    llm_with_tools
    | (lambda x: x.tool_calls[0]["args"])  # Extract args for add
    | add_numbers                          # Perform addition
    | (lambda result: {"x": result})       # Format result for squaring
    | square_number                        # Perform squaring
)

In [11]:
result = chain.invoke("What is the square of the sum of 44444 and 5?")
print("Final result:", result)


Final result: 1975713601


Using pipe-based tool chains in LangChain offers simplicity for fixed tasks but comes with significant limitations compared to agents. 

    These static chains follow a predefined sequence and lack the ability to reason dynamically, making them unsuitable for handling varied or complex user queries.
        
    They do not support decision-making, error handling, or contextual memory, and become difficult to scale as more tools are added. 
        
    Agents simplify using multiple tools by dynamically selecting and managing them based on the input. Instead of following a fixed sequence, agents decide which tool to use and how many times, offering flexibility.

#### Prompt Template  Pulling from LangChain 

In [13]:
# Get the prompt to use - can be replaced with any prompt that includes variables "agent_scratchpad" and "input"!
from langchain import hub
prompt = hub.pull("hwchase17/openai-tools-agent")
prompt.pretty_print()

================================ System Message ================================

You are a helpful assistant

============================= Messages Placeholder =============================

{chat_history}

================================ Human Message =================================

{input}

============================= Messages Placeholder =============================

{agent_scratchpad}


In [14]:
##

In [15]:
tools = [add_numbers, square_number]

In [16]:
# Construct the tool calling agent
agent = create_tool_calling_agent(llm, tools, prompt)

# Create an agent executor by passing in the agent and tools
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [17]:
agent_executor.invoke(
   {
       "input": "Write a poem on CS students in Indian Universities."
   }
)



> Entering new AgentExecutor chain...
[{'type': 'text', 'text': "I will write a poem about CS students in Indian universities.In India, there's a special breed,\nOf students who code with speed,\nThey're the CS crew, the tech-savvy few,\nIn universities, they're the ones who pursue.\n\nTheir minds are sharp, their skills refined,\nThey tackle bugs and don't resign,\nWith algorithms and data structures in hand,\nThey're ready to take on any demand.\n\nFrom coding marathons to hackathons,\nThey thrive on challenges, never frowns,\nTheir passion ignites, as they code through the night,\nDetermined to make their innovative mark bright.\n\nIn labs and libraries, they spend their days,\nLearning new languages in myriad ways,\nC, C++, Python, and more,\nTheir knowledge always eager to explore.\n\nThey dream of startups and tech innovations,\nChanging the world with their creations,\nFrom artificial intelligence to machine learning might,\nThey strive to make everything just right.\n\nSo her

{'input': 'Write a poem on CS students in Indian Universities.',
 'output': [{'type': 'text',
   'text': "I will write a poem about CS students in Indian universities.In India, there's a special breed,\nOf students who code with speed,\nThey're the CS crew, the tech-savvy few,\nIn universities, they're the ones who pursue.\n\nTheir minds are sharp, their skills refined,\nThey tackle bugs and don't resign,\nWith algorithms and data structures in hand,\nThey're ready to take on any demand.\n\nFrom coding marathons to hackathons,\nThey thrive on challenges, never frowns,\nTheir passion ignites, as they code through the night,\nDetermined to make their innovative mark bright.\n\nIn labs and libraries, they spend their days,\nLearning new languages in myriad ways,\nC, C++, Python, and more,\nTheir knowledge always eager to explore.\n\nThey dream of startups and tech innovations,\nChanging the world with their creations,\nFrom artificial intelligence to machine learning might,\nThey strive t

In [18]:
agent_executor.invoke(
   {
       "input": "add number 22, 33 and 44 and square 22 and 33 added together."
   }
)



> Entering new AgentExecutor chain...

Invoking: `add_numbers` with `{'a': 22, 'b': 33}`
responded: [{'type': 'text', 'text': "I will use the 'add_numbers' tool to add 22, 33 and 44. I will then use the 'square_number' tool to square the sum of 22 and 33.", 'index': 0}, {'type': 'tool_use', 'name': 'add_numbers', 'id': 'tooluse_0cvDTXhtEd1e27qYNddHIG', 'index': 1, 'input': '{\n    "a": 22,\n    "b": 33\n}'}, {'type': 'tool_use', 'name': 'add_numbers', 'id': 'tooluse_f3zdkCYsxPG3r6SCvStjv7', 'index': 2, 'input': '{\n    "a": 55,\n    "b": 44\n}'}, {'type': 'tool_use', 'name': 'square_number', 'id': 'tooluse_8AAeouQ9sRCM13PoxeUkp2', 'index': 3, 'input': '{\n    "x": 99\n}'}]

55
Invoking: `add_numbers` with `{'a': 55, 'b': 44}`
responded: [{'type': 'text', 'text': "I will use the 'add_numbers' tool to add 22, 33 and 44. I will then use the 'square_number' tool to square the sum of 22 and 33.", 'index': 0}, {'type': 'tool_use', 'name': 'add_numbers', 'id': 'tooluse_0cvDTXhtEd1e27qYNddHI

{'input': 'add number 22, 33 and 44 and square 22 and 33 added together.',
 'output': [{'type': 'text',
   'text': 'The sum of 22, 33 and 44 is **99**. The square of 22 and 33 added together is **9801**.',
   'index': 0}]}

In [15]:
agent_executor.invoke(
   {
       "input": "Add 55 with 22 and Find the square of the result."
   }
)



> Entering new AgentExecutor chain...

Invoking: `add_numbers` with `{'a': 55, 'b': 22}`
responded: [{'type': 'text', 'text': 'I will first add 55 and 22, then square the result.', 'index': 0}, {'type': 'tool_use', 'name': 'add_numbers', 'id': 'tooluse_eGSt89m48VsMY8fYdfDNZs', 'index': 1, 'input': '{\n    "a": 55,\n    "b": 22\n}'}]

77
Invoking: `square_number` with `{'x': 77}`
responded: [{'type': 'text', 'text': '55 + 22 = 77. Now I will square 77.', 'index': 0}, {'type': 'tool_use', 'name': 'square_number', 'id': 'tooluse_CyA4AUJsSzuq7w0ZT8sQ3u', 'index': 1, 'input': '{\n    "x": 77\n}'}]

5929[{'type': 'text', 'text': 'The square of 55 + 22 is **5929**.', 'index': 0}]

> Finished chain.


{'input': 'Add 55 with 22 and Find the square of the result.',
 'output': [{'type': 'text',
   'text': 'The square of 55 + 22 is **5929**.',
   'index': 0}]}

In [16]:
agent_executor.invoke(
   {
       "input": "Find the square of 245 and add -22 to the result."
   }
)



> Entering new AgentExecutor chain...

Invoking: `square_number` with `{'x': 245}`
responded: [{'type': 'text', 'text': 'I will first find the square of 245 and then add -22 to the result.', 'index': 0}, {'type': 'tool_use', 'name': 'square_number', 'id': 'tooluse_3h1E8FPTNSyPwcnDae6Cgz', 'index': 1, 'input': '{\n    "x": 245\n}'}]

60025
Invoking: `add_numbers` with `{'a': 60025, 'b': -22}`
responded: [{'type': 'text', 'text': 'The square of 245 is 60025. Now I will add -22 to this number.', 'index': 0}, {'type': 'tool_use', 'name': 'add_numbers', 'id': 'tooluse_doMhqkPjr9srM3DfVM9uMJ', 'index': 1, 'input': '{\n    "a": 60025,\n    "b": -22\n}'}]

60003[{'type': 'text', 'text': 'The answer is **60003**.', 'index': 0}]

> Finished chain.


{'input': 'Find the square of 245 and add -22 to the result.',
 'output': [{'type': 'text', 'text': 'The answer is **60003**.', 'index': 0}]}

####  What is agent_scratchpad?
It stores the agent's internal monologue:

    Tool calls
    
    Outputs from tools
    
    Reasoning steps

This data is fed back into the prompt to help the LLM reason more effectively in subsequent steps.





### `chat_history`
**Purpose:**  
Stores the full conversation between the user and the assistant.

**Usage:**  
- Helps maintain context across multiple turns.
- Ensures the assistant can refer back to previous messages.
- Useful for generating coherent and context-aware responses.

**Example Content:**
```python
[
    HumanMessage(content="What's the weather today?"),
    AIMessage(content="It's sunny in Mysore."),
    HumanMessage(content="What about tomorrow?")
]
```

---

###  `agent_scratchpad`
**Purpose:**  
Stores the internal reasoning steps or intermediate thoughts of the agent during a single task or query.

**Usage:**  
- Used when the agent is solving a complex problem and needs to "think out loud."
- Helps track tool calls, observations, and decisions made during the reasoning process.
- Not shown to the user directly, but helps the agent stay organized.

**Example Content:**
```python
[
    ToolCall(name="search_weather", args={"location": "Mysore"}),
    ToolResult(content="Tomorrow's forecast is rainy."),
    AIMessage(content="Based on the forecast, it will rain tomorrow.")
]
```

---

### Summary of Differences

| Feature            | `chat_history`                          | `agent_scratchpad`                        |
|-------------------|------------------------------------------|-------------------------------------------|
| Scope             | Entire conversation                     | Current reasoning process                 |
| Visibility        | Shown to user                           | Internal to agent                         |
| Content Type      | User and AI messages                    | Tool calls, thoughts, intermediate steps  |
| Purpose           | Maintain context                        | Track reasoning and tool usage            |

---
